In [ ]:
# Path setup – ensures module imports work from the notebooks/ subfolder
import sys
from pathlib import Path

# notebooks/ is one level below src/main/python; add the python root to path.
sys.path.insert(0, str(Path.cwd().parent))


In [1]:
# Import standard libraries
from pathlib import Path
import pandas as pd
from itables import show
import plotly.express as px
import numpy as np


In [2]:
# Import pipeline modules
from analysis import config
from analysis.pipeline import (
    load_summary_runs,
    compute_theoretical,
    compute_delay_quantiles,
    merge_all,
    JOIN_COLS,
)
from analysis.reports.summary_tables import export_summary_tables_html
from analysis.reports.confusion_matrix import export_confusion_matrix_html
from analysis.reports.max_volume_tables import export_max_volume_tables_html


In [3]:
# ============================================================
# CONFIG – edit analysis/config.py, or override here
# ============================================================

# Import defaults from config module
OUTPUT_ROOT = config.OUTPUT_ROOT
run_id      = config.RUN_ID
period      = config.PERIOD_S

# Quick override example (uncomment to use a different run):
# run_id = "output_20260224_it5_n1000"
# OUTPUT_ROOT = config.FILER / run_id

print(f"Analysing: {run_id}")
print(f"Root     : {OUTPUT_ROOT}")


In [4]:
# Step 1 – Load simulation summary
summary_runs_df = load_summary_runs(OUTPUT_ROOT / "output_run_summary.csv")


Found summary runs CSV file: /Users/nicolasdulex/devsbb/GZB_analysis/output_20260223_it5_n100/output_run_summary.csv. Will use it to compute the Link utilisation.


In [5]:
summary_runs_df

,use_case,building_block,run_id,operating_mode,product_mix,flow_pattern,volume,sample_index,analysis_window_utilization,analysis_window_delay_at_destination,analysis_window_headway_violation_tail_to_head,analysis_window_headway_violation_head_to_head,total_delay_at_destination,total_headway_violation_tail_to_head,total_headway_violation_head_to_head,trains_departed,trains_arrived,trains_stuck
0,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_005,express_pass,EXPRESS,PASS,8,5,0.3200,0.0,96.0,74.0,0.0,288.0,222.0,48,48,0
1,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_026,express_pass,EXPRESS,PASS,8,26,0.3200,0.0,118.0,74.0,0.0,354.0,222.0,48,48,0
2,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_027,express_pass,EXPRESS,PASS,8,27,0.3200,0.0,72.0,28.0,0.0,216.0,84.0,48,48,0
3,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_044,express_pass,EXPRESS,PASS,8,44,0.3200,0.0,56.0,34.0,0.0,168.0,102.0,48,48,0
4,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_075,express_pass,EXPRESS,PASS,8,75,0.3200,0.0,8.0,0.0,0.0,24.0,0.0,48,48,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22495,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_030,transit_trunk,TRANSIT,TRUNK,12,30,0.4192,3594.0,2844.0,2104.0,10611.0,8334.0,6168.0,144,144,0
22496,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_017,transit_trunk,TRANSIT,TRUNK,12,17,0.4336,3968.0,2730.0,1932.0,11904.0,8190.0,5796.0,144,144,0
22497,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_089,transit_trunk,TRANSIT,TRUNK,12,89,0.4233,4152.0,2822.0,2026.0,11917.0,8270.0,5948.0,144,144,0
22498,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_038,transit_trunk,TRANSIT,TRUNK,12,38,0.4222,4298.0,2704.0,1996.0,12237.0,7922.0,5851.0,144,144,0


In [6]:
# Step 2 – Compute theoretical utilisation
df_theorical = compute_theoretical(
    OUTPUT_ROOT,
    capacity_by_building_block=config.CAPACITY_BY_BUILDING_BLOCK,
    headway_overrides=config.HEADWAY_OVERRIDES,
)
df_theorical


[INFO] use_case='uc_0' | plan='uc0_operational_plan.json' | period_s=1800 | volumes=8..16 step=1 | building_blocks=2 | expected_rows~=18
[PROGRESS] use_case=uc_0 | blocks 2/2 | last_block=0.00s | avg~=0.00s/block | rows=18/18 | elapsed=00:00 | bb='uc0_bb2'
[INFO] use_case='uc_1' | plan='uc1_operational_plan.json' | period_s=1800 | volumes=4..12 step=1 | building_blocks=3 | expected_rows~=135
[PROGRESS] use_case=uc_1 | blocks 3/3 | last_block=0.00s | avg~=0.00s/block | rows=153/153 | elapsed=00:00 | bb='uc1_bb3'
[INFO] use_case='uc_2' | plan='uc2_operational_plan.json' | period_s=1800 | volumes=4..12 step=1 | building_blocks=2 | expected_rows~=72
[PROGRESS] use_case=uc_2 | blocks 2/2 | last_block=0.00s | avg~=0.00s/block | rows=225/225 | elapsed=00:00 | bb='uc2_bb2'


,use_case,building_block,operating_mode,volume,departures,transit_time_min,stop_time_min,total_time_min,utilisation_theorical
0,uc_0,uc0_bb1,express_pass,8,16.0,28.0,0.0,28.0,0.466667
1,uc_0,uc0_bb1,express_pass,9,18.0,31.5,0.0,31.5,0.525000
2,uc_0,uc0_bb1,express_pass,10,20.0,35.0,0.0,35.0,0.583333
3,uc_0,uc0_bb1,express_pass,11,22.0,38.5,0.0,38.5,0.641667
4,uc_0,uc0_bb1,express_pass,12,24.0,42.0,0.0,42.0,0.700000
...,...,...,...,...,...,...,...,...,...
220,uc_2,uc2_bb2,transit_trunk,8,16.0,43.0,0.0,43.0,0.716667
221,uc_2,uc2_bb2,transit_trunk,9,18.0,47.0,0.0,47.0,0.783333
222,uc_2,uc2_bb2,transit_trunk,10,20.0,53.0,0.0,53.0,0.883333
223,uc_2,uc2_bb2,transit_trunk,11,22.0,57.0,0.0,57.0,0.950000


In [7]:
JOIN_COLS = [
    "use_case",
    "building_block",
    "operating_mode",
    "volume",
]
missing_left  = set(JOIN_COLS) - set(summary_runs_df.columns)
missing_right = set(JOIN_COLS) - set(df_theorical.columns)

if missing_left or missing_right:
    raise ValueError(
        f"Missing columns — left: {missing_left}, right: {missing_right}"
    )

In [8]:
df_merged = summary_runs_df.merge(
    df_theorical,
    on=JOIN_COLS,
    how="inner",          # only combinations that exist in both
)
df_merged


,use_case,building_block,run_id,operating_mode,product_mix,flow_pattern,volume,sample_index,analysis_window_utilization,analysis_window_delay_at_destination,...,total_headway_violation_tail_to_head,total_headway_violation_head_to_head,trains_departed,trains_arrived,trains_stuck,departures,transit_time_min,stop_time_min,total_time_min,utilisation_theorical
0,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_005,express_pass,EXPRESS,PASS,8,5,0.3200,0.0,...,288.0,222.0,48,48,0,16.0,28.0,0.0,28.0,0.466667
1,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_026,express_pass,EXPRESS,PASS,8,26,0.3200,0.0,...,354.0,222.0,48,48,0,16.0,28.0,0.0,28.0,0.466667
2,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_027,express_pass,EXPRESS,PASS,8,27,0.3200,0.0,...,216.0,84.0,48,48,0,16.0,28.0,0.0,28.0,0.466667
3,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_044,express_pass,EXPRESS,PASS,8,44,0.3200,0.0,...,168.0,102.0,48,48,0,16.0,28.0,0.0,28.0,0.466667
4,uc_0,uc0_bb1,uc0_bb1_express_pass_volume_08_sample_075,express_pass,EXPRESS,PASS,8,75,0.3200,0.0,...,24.0,0.0,48,48,0,16.0,28.0,0.0,28.0,0.466667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22495,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_030,transit_trunk,TRANSIT,TRUNK,12,30,0.4192,3594.0,...,8334.0,6168.0,144,144,0,24.0,63.0,0.0,63.0,1.050000
22496,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_017,transit_trunk,TRANSIT,TRUNK,12,17,0.4336,3968.0,...,8190.0,5796.0,144,144,0,24.0,63.0,0.0,63.0,1.050000
22497,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_089,transit_trunk,TRANSIT,TRUNK,12,89,0.4233,4152.0,...,8270.0,5948.0,144,144,0,24.0,63.0,0.0,63.0,1.050000
22498,uc_2,uc2_bb2,uc2_bb2_transit_trunk_volume_12_sample_038,transit_trunk,TRANSIT,TRUNK,12,38,0.4222,4298.0,...,7922.0,5851.0,144,144,0,24.0,63.0,0.0,63.0,1.050000


In [10]:
fig = px.scatter(
    df_merged,
    x="utilisation_theorical",
    y="analysis_window_utilization",
    color="use_case",
    opacity=0.6,
    hover_data=[
        "use_case",
        "building_block",
        "operating_mode",
        "volume",
        "sample_index",
    ],
)
xmin = min(df_merged["utilisation_theorical"].min(), df_merged["analysis_window_utilization"].min())
xmax = max(df_merged["utilisation_theorical"].max(), df_merged["analysis_window_utilization"].max())

fig.add_shape(
    type="line",
    x0=xmin, y0=xmin,
    x1=xmax, y1=xmax,
    line=dict(dash="dash"),
)

fig.update_layout(
    xaxis_title="utilisation_theorical",
    yaxis_title="analysis_window_utilization",
)


fig.show()


In [11]:
for uc, g in df_merged.groupby("use_case", dropna=False):

    fig = px.scatter(
        g,
        x="utilisation_theorical",
        y="analysis_window_utilization",
        opacity=0.6,
        title=f"use_case = {uc}",
    )

    xmin = min(g["utilisation_theorical"].min(), g["analysis_window_utilization"].min())
    xmax = max(g["utilisation_theorical"].max(), g["analysis_window_utilization"].max())

    fig.add_shape(
        type="line",
        x0=xmin, y0=xmin,
        x1=xmax, y1=xmax,
        line=dict(dash="dash"),
    )

    fig.show()


In [12]:
# ============================================================
# ANALYSIS 2 : Delay quantiles vs utilisation (theorical and linkstates)
# ============================================================

In [36]:
# Step 3 – Compute delay quantiles
df_q = compute_delay_quantiles(
    OUTPUT_ROOT,
    period_s=config.PERIOD_S,
    quantiles_pct=config.QUANTILES_PCT,
)
df_q


,use_case,building_block,operating_mode,volume,train_volume_per_hour,q0,q2,q5,q10,q50,q90,q100,utilization_mean,utilization_median
0,uc_0,uc0_bb1,express_pass,8,16.0,0.000000,0.000000,0.000000,0.362500,8.125000,23.437500,44.000000,0.304316,0.30305
1,uc_0,uc0_bb1,express_pass,9,18.0,0.000000,0.544444,1.322222,2.000000,9.333333,25.344444,40.777778,0.341225,0.34170
2,uc_0,uc0_bb1,express_pass,10,20.0,0.000000,0.898000,1.895000,3.100000,11.550000,28.010000,61.800000,0.377640,0.37805
3,uc_0,uc0_bb1,express_pass,11,22.0,0.454545,0.816364,2.077273,3.681818,15.681818,35.118182,77.272727,0.414807,0.41640
4,uc_0,uc0_bb1,express_pass,12,24.0,0.666667,1.828333,3.212500,5.166667,16.291667,41.025000,76.666667,0.449317,0.45110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,uc_2,uc2_bb2,transit_trunk,8,16.0,0.875000,7.792500,10.281250,13.050000,34.125000,74.362500,136.000000,0.277759,0.27830
221,uc_2,uc2_bb2,transit_trunk,9,18.0,11.666667,13.197778,15.188889,18.500000,45.888889,91.333333,186.611111,0.309642,0.31015
222,uc_2,uc2_bb2,transit_trunk,10,20.0,13.500000,17.450000,21.495000,25.160000,54.250000,107.085000,278.300000,0.344316,0.34470
223,uc_2,uc2_bb2,transit_trunk,11,22.0,13.909091,26.916364,32.650000,35.000000,62.000000,127.513636,179.727273,0.383973,0.38295


In [14]:
# Step 4 – Merge delay quantiles with theoretical utilisation
df_merged_delay = merge_all(df_q, df_theorical)
df_merged_delay


,use_case,building_block,operating_mode,volume,train_volume_per_hour,q0,q2,q5,q10,q50,q90,q100,utilization_mean,utilization_median,departures,transit_time_min,stop_time_min,total_time_min,utilisation_theorical
0,uc_0,uc0_bb1,express_pass,8,16.0,0.000000,0.000000,0.000000,0.362500,8.125000,23.437500,44.000000,0.304316,0.30305,16.0,28.0,0.0,28.0,0.466667
1,uc_0,uc0_bb1,express_pass,9,18.0,0.000000,0.544444,1.322222,2.000000,9.333333,25.344444,40.777778,0.341225,0.34170,18.0,31.5,0.0,31.5,0.525000
2,uc_0,uc0_bb1,express_pass,10,20.0,0.000000,0.898000,1.895000,3.100000,11.550000,28.010000,61.800000,0.377640,0.37805,20.0,35.0,0.0,35.0,0.583333
3,uc_0,uc0_bb1,express_pass,11,22.0,0.454545,0.816364,2.077273,3.681818,15.681818,35.118182,77.272727,0.414807,0.41640,22.0,38.5,0.0,38.5,0.641667
4,uc_0,uc0_bb1,express_pass,12,24.0,0.666667,1.828333,3.212500,5.166667,16.291667,41.025000,76.666667,0.449317,0.45110,24.0,42.0,0.0,42.0,0.700000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,uc_2,uc2_bb2,transit_trunk,8,16.0,0.875000,7.792500,10.281250,13.050000,34.125000,74.362500,136.000000,0.277759,0.27830,16.0,43.0,0.0,43.0,0.716667
221,uc_2,uc2_bb2,transit_trunk,9,18.0,11.666667,13.197778,15.188889,18.500000,45.888889,91.333333,186.611111,0.309642,0.31015,18.0,47.0,0.0,47.0,0.783333
222,uc_2,uc2_bb2,transit_trunk,10,20.0,13.500000,17.450000,21.495000,25.160000,54.250000,107.085000,278.300000,0.344316,0.34470,20.0,53.0,0.0,53.0,0.883333
223,uc_2,uc2_bb2,transit_trunk,11,22.0,13.909091,26.916364,32.650000,35.000000,62.000000,127.513636,179.727273,0.383973,0.38295,22.0,57.0,0.0,57.0,0.950000


In [15]:
QUANTILES = ["q0", "q2", "q5", "q10", "q50"]

df_long = df_merged_delay.melt(
    id_vars=[
        "use_case",
        "building_block",
        "operating_mode",
        "utilisation_theorical",
        "volume",
    ],
    value_vars=QUANTILES,
    var_name="quantile",
    value_name="delay",
)


In [17]:
def categorize_delay(x):
    if pd.isna(x):
        return np.nan
    if x == 0:
        return "0"
    elif 0 < x < 15:
        return "0-15"
    else:
        return ">15"


In [18]:
df_long["delay_category"] = df_long["delay"].apply(categorize_delay)


In [19]:
show(df_merged_delay)

Loading ITables v2.7.0 from the internet... (need help?)


In [20]:
DELAY_CAT_ORDER = ["0","0-15", ">15"]

df_long["delay_category"] = pd.Categorical(
    df_long["delay_category"],
    categories=DELAY_CAT_ORDER,
    ordered=True,
)


In [21]:
df_long["delay_category"].value_counts(dropna=False)


delay_category
0-15    493
>15     406
0       226
Name: count, dtype: int64

In [28]:
df_merged_delay["volume"] = df_merged_delay["volume"]*(3600/period) #

In [29]:
# export_summary_tables_html is defined in analysis/reports/summary_tables.py
# ROW_CONFIG (KPI definitions) is also there — edit that file to change KPIs.


In [30]:
# Step 5a – Export summary tables
export_summary_tables_html(
    df_merged_delay,
    run_id,
    thresholds_by_operating_mode=config.THRESHOLDS_BY_OPERATING_MODE,
    default_utilisation_threshold=config.DEFAULT_UTILISATION_THRESHOLD,
    output_path=OUTPUT_ROOT / "summary_tables.html",
)


Saved to /Users/nicolasdulex/devsbb/GZB_analysis/output_20260223_it5_n100/summary_tables.html


In [31]:
# export_confusion_matrix_html is defined in analysis/reports/confusion_matrix.py
# KPI_CONFIG and _matrix_html helpers are also there.


In [32]:
# Step 5b – Export confusion matrix
export_confusion_matrix_html(
    df_merged_delay,
    run_id,
    thresholds_by_operating_mode=config.THRESHOLDS_BY_OPERATING_MODE,
    default_utilisation_threshold=config.DEFAULT_UTILISATION_THRESHOLD,
    output_path=OUTPUT_ROOT / "confusion_matrix.html",
)


Saved to /Users/nicolasdulex/devsbb/GZB_analysis/output_20260223_it5_n100/confusion_matrix.html


In [33]:
# export_max_volume_tables_html is defined in analysis/reports/max_volume_tables.py


In [34]:
# Step 5c – Export max volume tables
kpi = "q5"
export_max_volume_tables_html(
    df_merged_delay,
    run_id=run_id,
    kpi=kpi,
    threshold=5,
    thresholds_by_operating_mode=config.THRESHOLDS_BY_OPERATING_MODE,
    default_utilisation_threshold=config.DEFAULT_UTILISATION_THRESHOLD,
    output_path=OUTPUT_ROOT / f"max_volume_tables_{kpi}.html",
)


Saved to /Users/nicolasdulex/devsbb/GZB_analysis/output_20260223_it5_n100/max_volume_tables_q5.html
